# Exercise 11.1: Z → μμ with CMS open data

From *Programming in High Energy Particle Physics*, Chapter 11

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch11/ex11_1_solution.ipynb)

Exercise 11.1 Z → μμ with CMS open data Download a public CMS NanoAOD file and implement a complete event selection for \(Z \to \mu^+\mu^-\). Measure the Z boson mass and compare with the PDG value. Compute the statistical uncertainty and estimate the dominant systematic uncertainties.

<details><summary>Hint</summary>

Run 2 (2016) CMS NanoAOD data and simulation are public on the CERN Open Data portal , together with the matching Golden JSON. Read Muon_pt , Muon_eta , Muon_phi , Muon_charge and Muon_tightId with uproot, apply the Golden JSON with certified_mask from Listing 11.1, select two tight opposite-charge muons, and build the invariant mass with vector . Fit the peak with a Breit–Wigner convolved with a Gaussian (a Voigtian) plus a falling background. For the systematics, vary the muon momentum scale by its quoted uncertainty and the fit range and model.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q uproot awkward vector scipy matplotlib numpy")

**Data limitation:** no small stable CMS NanoAOD URL and matching Golden JSON is asserted here. The default writes a clearly labelled toy CMS-like ROOT file; real open-data analysis requires setting a verified URL and its certification file. The toy fit does not establish a CMS mass measurement.

### Step 1: Load CMS-like NanoAOD branches and apply a certified mask

In [ ]:
import numpy as np, awkward as ak, uproot
from scipy.special import voigt_profile
rng=np.random.default_rng(42); N=80000; MZ=91.1876
NANOAOD_URL=''  # set to a verified public HTTPS CMS NanoAOD file
GOLDEN_JSON_URL=''  # matching certified JSON from the same data-taking period
if not NANOAOD_URL:
    mass=np.clip(MZ+rng.standard_cauchy(N)*1.25+rng.normal(0,1.3,N),65,120)
    is_background=rng.random(N)<.04; mass[is_background]=rng.uniform(70,110,is_background.sum())
    pt=mass/2
    with uproot.recreate('toy_cms_nano.root') as f:
        f['Events']={'Muon_pt':ak.Array(np.stack([pt,pt],axis=1)),
                     'Muon_eta':ak.Array(np.zeros((N,2))),
                     'Muon_phi':ak.Array(np.tile([0.,np.pi],(N,1))),
                     'Muon_charge':ak.Array(np.tile([1,-1],(N,1))),
                     'Muon_tightId':ak.Array(np.ones((N,2),dtype=bool)),
                     'run':np.full(N,367112),'luminosityBlock':rng.integers(1,101,N)}
    source='toy_cms_nano.root'; golden={'367112':[[1,100]]}
else:
    if not (NANOAOD_URL.startswith('https://') and GOLDEN_JSON_URL.startswith('https://')):
        raise ValueError('Supply HTTPS URLs for the NanoAOD and matching Golden JSON')
    import json, urllib.request
    with urllib.request.urlopen(GOLDEN_JSON_URL) as response: golden=json.load(response)
    source=NANOAOD_URL
branches=['Muon_pt','Muon_eta','Muon_phi','Muon_charge','Muon_tightId','run','luminosityBlock']
events=uproot.open(source)['Events'].arrays(branches,library='ak'); print('Source:',source)

### Step 2: Select tight opposite-charge muon pairs

In [ ]:
import vector
vector.register_awkward()
def certified_mask(run,ls,golden):
    good=np.array([(int(r)<<20)|section for r,ranges in golden.items()
                   for lo,hi in ranges for section in range(lo,hi+1)],dtype=np.int64)
    keys=(np.asarray(run,dtype=np.int64)<<20)|np.asarray(ls,dtype=np.int64)
    return np.isin(keys,good)
events=events[certified_mask(events.run,events.luminosityBlock,golden)]
mu=ak.zip({'pt':events.Muon_pt,'eta':events.Muon_eta,'phi':events.Muon_phi,
           'mass':ak.ones_like(events.Muon_pt)*.105658,'charge':events.Muon_charge,
           'tight':events.Muon_tightId},with_name='Momentum4D')
mu=mu[(mu.pt>25)&(abs(mu.eta)<2.4)&mu.tight]
pairs=ak.combinations(mu,2,fields=['m1','m2'])
pairs=pairs[pairs.m1.charge*pairs.m2.charge<0]
masses=ak.to_numpy(ak.flatten((pairs.m1+pairs.m2).mass)); masses=masses[(masses>70)&(masses<110)]
assert len(masses)>1000

### Step 3: Fit a Voigtian and estimate uncertainty

In [ ]:
from scipy.optimize import curve_fit
from scipy.special import voigt_profile
import matplotlib.pyplot as plt
bins=np.linspace(70,110,81); counts,_=np.histogram(masses,bins); centers=(bins[1:]+bins[:-1])/2
width=bins[1]-bins[0]
def model(x,amp,loc,sigma,bkg): return amp*voigt_profile(x-loc,sigma,1.25)*width+bkg
fit,cov=curve_fit(model,centers,counts,p0=[len(masses),91.2,1.3,5],sigma=np.sqrt(np.maximum(counts,1)),absolute_sigma=True,
                  bounds=([0,85,.2,0],[np.inf,98,5,np.inf]))
stat=np.sqrt(cov[1,1]); scale_shift=fit[1]*.001  # illustrative 0.1% momentum-scale shift, not an official calibration
fit2,_=curve_fit(model,centers[(centers>75)&(centers<105)],counts[(centers>75)&(centers<105)],p0=fit,
                 sigma=np.sqrt(np.maximum(counts[(centers>75)&(centers<105)],1)),absolute_sigma=True,
                 bounds=([0,85,.2,0],[np.inf,98,5,np.inf]))
print(f'Voigt peak: {fit[1]:.4f} ± {stat:.4f} GeV (stat); PDG {MZ:.4f} GeV')
print(f'Illustrative 0.1% momentum scale shift: {scale_shift:.3f} GeV; fit-range shift: {fit2[1]-fit[1]:+.3f} GeV')
fig,ax=plt.subplots(figsize=(8,4)); ax.errorbar(centers,counts,np.sqrt(counts),fmt='.',label='toy selected events')
ax.plot(centers,model(centers,*fit),label='Voigt + flat background'); ax.axvline(MZ,color='black',ls='--',label='PDG')
ax.set(xlabel=r'$m_{\mu\mu}$ [GeV]',ylabel='Events / 0.5 GeV'); ax.legend(); plt.show()

The book gives m_Z = 91.1876 GeV. For a few hundred thousand selected events, statistical precision can reach a few MeV, but uncorrected real CMS data can shift by tens to 100 MeV; muon momentum scale and fit model dominate. The printed toy uncertainties and illustrative scale variation are not CMS estimates.

**Book answer:** The PDG value is \(m_Z = 91.1876\) GeV. With a few hundred thousand selected events the statistical uncertainty on the fitted peak position is at the level of a few MeV, far below the systematic effects: without the official muon momentum corrections the peak can be shifted by tens to a hundred MeV, and the muon momentum scale, together with the choice of fit model, dominates the uncertainty.